In [ ]:
#!/usr/bin/env python3
import os
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models, datasets
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import classification_report
from torchvision.models import resnet50, ResNet50_Weights
import matplotlib.pyplot as plt

# ====================== CONFIG ======================
DATA_DIR = './Aerial_Landscapes'  # 数据目录：子目录应该是 15 个类别的文件夹
BATCH_SIZE = 32
NUM_EPOCHS = 10
NUM_CLASSES = 15
IMAGE_SIZE = 224
LEARNING_RATE = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# =====================================================

# ============ 1. 数据预处理与加载 ============
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ResNet 的 ImageNet 标准
                         std=[0.229, 0.224, 0.225])
])

# 加载整个数据集（假设数据目录结构是 ImageFolder 格式）
full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)

# 训练集 & 验证集划分（80% / 20%）
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# ============ 2. 加载 ResNet 模型 ============
model = resnet50(weights=ResNet50_Weights.DEFAULT)

# 替换最后的全连接层
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(DEVICE)

# ============ 3. 定义损失函数和优化器 ============
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# ============ 4. 训练与验证函数 ============
def train(model, loader):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

def evaluate(model, loader):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return running_loss / total, correct / total, all_preds, all_labels

# ============ 5. 模型训练主循环 ============
train_accs, val_accs = [], []
for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train(model, train_loader)
    val_loss, val_acc, _, _ = evaluate(model, val_loader)

    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f}, Acc: {val_acc:.4f}")
    print("-" * 40)

# ============ 6. 模型评估与保存 ============
_, _, all_preds, all_labels = evaluate(model, val_loader)
print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=full_dataset.classes))

# 保存模型
torch.save(model.state_dict(), 'resnet50_skyview.pth')

# 可视化准确率变化
plt.plot(train_accs, label='Train Acc')
plt.plot(val_accs, label='Val Acc')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()
